# Self-supervised contrastive learning for label-efficient chest X-ray classification

**Colab runner: data prep → 5 SSL methods → baselines → label-budget sweep.** Methods: SimCLR, MoCo v2, BYOL, NNCLR, SwAV (all hand-written PyTorch, same encoder / data / augmentations), plus supervised-from-scratch and ImageNet reference baselines.

**How to use**
1. `Runtime → Change runtime type → GPU`.
2. Put **either** `kaggle.json` **or** the dataset `archive.zip` in `MyDrive/chexpert_ssl/` (Drive folder is created on first run).
3. Run cells top to bottom. Every cell is safe to re-run after a disconnect: preprocessing, splits, pretraining (per-epoch checkpoints) and downstream runs all resume or skip what is finished.
4. Run the **smoke tests** before any cell marked `[FULL]`. They print seconds/step so you can size the epoch counts.
5. The official `valid.csv` is only touched in the last cell, behind an explicit confirmation flag.

Design choices (change in the config cell): ResNet-18 at 128 px (Colab-sized; set `arch="resnet50"`, `train_res=224` if you get the compute), images cached once as uint8 arrays, augmentations run on the GPU.

In [ ]:
import os, sys, json, time, math, random, shutil, subprocess, re, copy, threading, queue, warnings
from pathlib import Path
import numpy as np
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = Path("/content/drive/MyDrive/chexpert_ssl")
    LOCAL = Path("/content/work")
else:
    PROJECT = Path(os.environ.get("PROJECT_DIR", "./chexpert_ssl")).resolve()
    LOCAL = Path(os.environ.get("LOCAL_DIR", "./work")).resolve()
PROJECT.mkdir(parents=True, exist_ok=True)
LOCAL.mkdir(parents=True, exist_ok=True)

import torch, torch.nn as nn, torch.nn.functional as F, torchvision
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = DEVICE == "cuda"
print("torch", torch.__version__, "| torchvision", torchvision.__version__, "| device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0), f"| {torch.cuda.get_device_properties(0).total_memory/2**30:.1f} GB")
else:
    print("WARNING: no GPU. In Colab use Runtime > Change runtime type > GPU.")

In [ ]:
CFG = dict(
    labels=["Atelectasis", "Cardiomegaly", "Consolidation", "Edema", "Pleural Effusion"],
    uncertain_to_one=["Atelectasis", "Edema"],   # U-Ones for these two; U-Zeros for the other three
    store_res=128,            # resolution the images are cached at
    train_res=128,            # resolution fed to the network (<= store_res)
    arch="resnet18",          # "resnet18" or "resnet50"
    dev_fraction=0.10, split_seed=1234,
    seeds=[0, 1, 2], budgets=[0.01, 0.05, 0.10, 0.25, 1.0],
    max_images=None,          # e.g. 5000 -> dry run on a random sample of train.csv (separate artifact folder)
    ssl=dict(batch_size=256, epochs=30, warmup_epochs=2, lr=1e-3, weight_decay=1e-4, seed=0,
             temperature=dict(simclr=0.2, moco=0.2, nnclr=0.1, swav=0.1),
             moco_queue=4096, moco_momentum=0.99, moco_bn_splits=4,
             byol_tau=0.99, nnclr_queue=8192, swav_prototypes=300),
    down=dict(batch_size=64, patience=7, weight_decay=1e-4, lr_head=1e-3,
              lr_backbone_pretrained=1e-4, lr_backbone_scratch=1e-3,
              linear_epochs=100, linear_patience=10,
              max_epochs={0.01: 30, 0.05: 30, 0.10: 25, 0.25: 20, 1.0: 10},
              pos_weight_max=50.0),
)
LABELS = CFG["labels"]
Y_COLS = [f"y_{l}" for l in LABELS]
RES_STORE, RES_IN = CFG["store_res"], CFG["train_res"]
assert RES_IN <= RES_STORE

TAG = f"r{RES_STORE}" + (f"_n{CFG['max_images']}" if CFG["max_images"] else "")
ART = PROJECT / TAG        # Drive: manifests, splits, checkpoints, results (persistent)
LOC = LOCAL / TAG          # local disk: cached arrays (fast, per session)
for sub in ["manifests", "splits", "arrays", "pretrain", "downstream", "reports"]:
    (ART / sub).mkdir(parents=True, exist_ok=True)
LOC.mkdir(parents=True, exist_ok=True)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
print("artifacts:", ART)

In [ ]:
# ---- locate (or download) the dataset ----
DATA_DIR = LOCAL / "data"

def find_csv_dir():
    if not DATA_DIR.exists(): return None
    for p in DATA_DIR.rglob("train.csv"):
        if (p.parent / "valid.csv").exists(): return p.parent
    return None

CSV_DIR = find_csv_dir()
if CSV_DIR is None:
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    zip_on_drive = PROJECT / "archive.zip"
    if zip_on_drive.exists():
        print("unzipping", zip_on_drive)
        subprocess.run(["unzip", "-q", "-o", str(zip_on_drive), "-d", str(DATA_DIR)], check=True)
    else:
        kj = PROJECT / "kaggle.json"
        assert kj.exists(), f"Put kaggle.json (or archive.zip) in {PROJECT}"
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"], check=True)
        os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
        shutil.copy(kj, os.path.expanduser("~/.kaggle/kaggle.json"))
        os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
        subprocess.run(["kaggle", "datasets", "download", "-d", "mimsadiislam/chexpert",
                        "-p", str(DATA_DIR), "--unzip"], check=True)
    CSV_DIR = find_csv_dir()
assert CSV_DIR is not None, "train.csv / valid.csv not found after download"
print("CSV dir:", CSV_DIR)

# CheXpert CSV paths look like 'CheXpert-v1.0-small/train/patient00001/study1/view1_frontal.jpg'.
# Work out once how they map onto the folder layout we actually have.
_first = pd.read_csv(CSV_DIR / "train.csv", nrows=1)["Path"].iloc[0]
_options = [lambda p: CSV_DIR.parent / p,
            lambda p: CSV_DIR / p,
            lambda p: CSV_DIR / p.split("/", 1)[1]]
RESOLVE = next((f for f in _options if f(_first).exists()), None)
assert RESOLVE is not None, f"cannot resolve image path {_first!r} under {CSV_DIR}"
print("example image:", RESOLVE(_first))

In [ ]:
# ---- label mapping, manifests, unit tests ----
def map_uncertain(raw, one_if_uncertain):
    # 1 -> 1, 0 -> 0, blank -> 0, -1 (uncertain) -> 1 (U-Ones) or 0 (U-Zeros)
    t = pd.Series(raw, dtype="float64").fillna(0.0)
    t = t.where(t != -1, 1.0 if one_if_uncertain else 0.0)
    return t.astype(np.float32)

def resolve_targets(df):
    out = pd.DataFrame(index=df.index)
    for l in LABELS:
        out[f"raw_{l}"] = df[l]
        out[f"y_{l}"] = map_uncertain(df[l], l in CFG["uncertain_to_one"])
    return out

assert map_uncertain([1, 0, -1, np.nan], True).tolist() == [1, 0, 1, 0]
assert map_uncertain([1, 0, -1, np.nan], False).tolist() == [1, 0, 0, 0]
print("label-mapping tests passed")

def build_manifest(csv_path, partition):
    df = pd.read_csv(csv_path)
    missing_cols = [c for c in LABELS + ["Path", "Frontal/Lateral", "AP/PA"] if c not in df.columns]
    assert not missing_cols, f"missing columns: {missing_cols}"
    if CFG["max_images"] and partition == "train":
        df = df.sample(n=min(CFG["max_images"], len(df)), random_state=0).sort_index()
    df = df.reset_index(drop=True)
    m = pd.DataFrame({
        "path": df["Path"],
        "patient_id": df["Path"].str.extract(r"patient(\d+)")[0].astype(int),
        "study_id": df["Path"].str.extract(r"study(\d+)")[0].astype(int),
        "sex": df.get("Sex"), "age": df.get("Age"),
        "view": df["Frontal/Lateral"], "ap_pa": df["AP/PA"],
    })
    m = pd.concat([m, resolve_targets(df)], axis=1)
    m["frontal"] = (m["view"] == "Frontal") & m["ap_pa"].isin(["AP", "PA"])
    m["partition"] = partition
    m["arr_idx"] = np.arange(len(m))
    return m

TRAIN_M = build_manifest(CSV_DIR / "train.csv", "train")
VALID_M = build_manifest(CSV_DIR / "valid.csv", "valid")
print(f"train rows: {len(TRAIN_M):,} | valid rows: {len(VALID_M):,}")
print(f"train patients: {TRAIN_M.patient_id.nunique():,} | valid patients: {VALID_M.patient_id.nunique():,}")
print("raw label values seen:", sorted(pd.concat([TRAIN_M[f'raw_{l}'] for l in LABELS]).dropna().unique()))

In [ ]:
# ---- cache every image once as a small uint8 array (resumable; copies persist on Drive) ----
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

def _read(args):
    i, fp, res = args
    try:
        with Image.open(fp) as im:
            im = im.convert("L").resize((res, res), Image.BILINEAR)   # grayscale; 3 channels are made in the model
            return i, np.asarray(im, dtype=np.uint8), True
    except Exception:
        return i, None, False

def preprocess(m, split):
    d_arr, d_ok = ART / "arrays" / f"{split}.npy", ART / "arrays" / f"{split}_ok.npy"
    l_arr = LOC / f"{split}.npy"
    if d_arr.exists() and d_ok.exists():
        if not l_arr.exists():
            print(f"[{split}] copying cached array from Drive ...")
            shutil.copyfile(d_arr, l_arr)
        return np.load(d_ok)
    N = len(m)
    arr = np.lib.format.open_memmap(l_arr, mode="w+", dtype=np.uint8, shape=(N, RES_STORE, RES_STORE))
    ok = np.zeros(N, dtype=bool)
    jobs = ((i, str(RESOLVE(p)), RES_STORE) for i, p in enumerate(m["path"]))
    with ThreadPoolExecutor(max_workers=min(16, 2 * (os.cpu_count() or 2))) as ex:
        for i, a, good in tqdm(ex.map(_read, jobs), total=N, desc=f"[{split}] reading", mininterval=5):
            if good: arr[i] = a; ok[i] = True
    arr.flush(); del arr
    tmp = d_arr.with_suffix(".tmp")
    shutil.copyfile(l_arr, tmp); tmp.replace(d_arr)     # array first, ok-mask last => both present means complete
    np.save(d_ok, ok)
    return ok

TRAIN_M["ok"] = preprocess(TRAIN_M, "train")
VALID_M["ok"] = preprocess(VALID_M, "valid")

TRAIN_M.to_csv(ART / "manifests" / "train_manifest.csv", index=False)
VALID_M.to_csv(ART / "manifests" / "final_validation_manifest.csv", index=False)
report = {s: dict(rows=int(len(m)), unreadable=int((~m.ok).sum()), frontal_ok=int((m.ok & m.frontal).sum()),
                  unreadable_paths=m.loc[~m.ok, "path"].tolist()) for s, m in [("train", TRAIN_M), ("valid", VALID_M)]}
(ART / "manifests" / "data_quality_report.json").write_text(json.dumps(report, indent=1))
print({s: {k: v for k, v in r.items() if k != "unreadable_paths"} for s, r in report.items()})

def get_array(split):
    return np.load(LOC / f"{split}.npy", mmap_mode="r")

In [ ]:
# ---- patient-level splits (dev split, pretraining pool, nested label budgets) ----
def n_budget(frac, n):
    return max(1, int(math.ceil(frac * n)))

split_file = ART / "splits" / "splits.json"
ok_train = TRAIN_M[TRAIN_M.ok]
if split_file.exists():
    SPLITS = json.loads(split_file.read_text())
else:
    patients = np.array(sorted(ok_train.patient_id.unique()))
    perm = np.random.RandomState(CFG["split_seed"]).permutation(patients)
    n_dev = max(1, int(round(CFG["dev_fraction"] * len(perm))))
    dev_p, tr_p = np.sort(perm[:n_dev]), np.sort(perm[n_dev:])
    fr_p = np.array(sorted(set(ok_train[ok_train.frontal & ok_train.patient_id.isin(tr_p)].patient_id)))
    SPLITS = dict(dev_patients=[int(x) for x in dev_p], train_patients=[int(x) for x in tr_p],
                  train_frontal_patients=[int(x) for x in fr_p], orders={})
# nested budget order per seed: budget k uses the first ceil(k * n) patients of the seed's permutation
for s in CFG["seeds"]:
    if str(s) not in SPLITS["orders"]:
        order = np.random.RandomState(s).permutation(np.array(SPLITS["train_frontal_patients"]))
        SPLITS["orders"][str(s)] = [int(x) for x in order]
split_file.write_text(json.dumps(SPLITS))

dev_set, tr_set = set(SPLITS["dev_patients"]), set(SPLITS["train_patients"])
val_set = set(VALID_M.patient_id)
assert not (dev_set & tr_set), "dev/train patient overlap"
assert not (val_set & (dev_set | tr_set)), "official-validation patients also appear in train.csv"
for s, order in SPLITS["orders"].items():
    assert set(order) <= tr_set
    sizes = [n_budget(f, len(order)) for f in CFG["budgets"]]
    assert sizes == sorted(sizes), "budgets must be nested"
print("patient-disjointness checks passed")

PRE_IDX = ok_train[ok_train.patient_id.isin(tr_set)].arr_idx.values                 # all views, no labels used
TRAIN_F = ok_train[ok_train.frontal & ok_train.patient_id.isin(tr_set)].reset_index(drop=True)
DEV_F = ok_train[ok_train.frontal & ok_train.patient_id.isin(dev_set)].reset_index(drop=True)
VALID_F = VALID_M[VALID_M.ok & VALID_M.frontal].reset_index(drop=True)
print(f"pretraining images: {len(PRE_IDX):,} | downstream train (frontal): {len(TRAIN_F):,} | "
      f"dev (frontal): {len(DEV_F):,} | official valid (frontal): {len(VALID_F):,}")
prev = pd.DataFrame({"train_frontal": TRAIN_F[Y_COLS].mean().values, "dev_frontal": DEV_F[Y_COLS].mean().values,
                     "valid_frontal": VALID_F[Y_COLS].mean().values}, index=LABELS).round(3)
print("label prevalence after mapping:"); print(prev)

In [ ]:
# ---- GPU augmentations (conservative, X-ray safe) ----
AUG_SSL = dict(scale=(0.6, 1.0), ratio=(0.85, 1.18), rot=10.0, trans=0.05, brightness=0.1, contrast=0.2,
               blur_p=0.5, blur_sigma=(0.1, 1.5), noise_p=0.5, noise_std=0.02, hflip=False)
AUG_DOWN = dict(scale=(0.8, 1.0), ratio=(0.9, 1.1), rot=5.0, trans=0.03, brightness=0.1, contrast=0.1,
                blur_p=0.0, blur_sigma=(0.1, 1.5), noise_p=0.0, noise_std=0.0, hflip=False)

def _u(n, lo, hi, dev):
    return torch.rand(n, device=dev) * (hi - lo) + lo

def gpu_augment(xb, res, a):
    # xb: uint8 (B,H,W) on device -> float (B,1,res,res) in [0,1]. No hue/saturation, no cutout, no flip by default.
    dev, B = xb.device, xb.shape[0]
    x = xb.float().div_(255.0).unsqueeze(1)
    area = _u(B, *a["scale"], dev)
    r = torch.exp(_u(B, math.log(a["ratio"][0]), math.log(a["ratio"][1]), dev))
    w = torch.sqrt(area * r).clamp(max=1.0)
    h = torch.sqrt(area / r).clamp(max=1.0)
    ang = _u(B, -a["rot"], a["rot"], dev) * math.pi / 180.0
    tx = (torch.rand(B, device=dev) * 2 - 1) * (1 - w) + (torch.rand(B, device=dev) * 2 - 1) * a["trans"] * 2
    ty = (torch.rand(B, device=dev) * 2 - 1) * (1 - h) + (torch.rand(B, device=dev) * 2 - 1) * a["trans"] * 2
    cos, sin = torch.cos(ang), torch.sin(ang)
    sgn = torch.ones(B, device=dev)
    if a["hflip"]:
        sgn = torch.where(torch.rand(B, device=dev) < 0.5, -sgn, sgn)
    theta = torch.stack([torch.stack([cos * w * sgn, -sin * h, tx], 1),
                         torch.stack([sin * w * sgn, cos * h, ty], 1)], 1)           # (B,2,3)
    grid = F.affine_grid(theta, (B, 1, res, res), align_corners=False)
    x = F.grid_sample(x, grid, mode="bilinear", padding_mode="border", align_corners=False)
    # brightness / contrast
    c = 1 + _u(B, -a["contrast"], a["contrast"], dev).view(B, 1, 1, 1)
    b = _u(B, -a["brightness"], a["brightness"], dev).view(B, 1, 1, 1)
    m = x.mean(dim=(1, 2, 3), keepdim=True)
    x = ((x - m) * c + m + b).clamp(0, 1)
    # gaussian blur with per-sample sigma (separable, grouped conv)
    if a["blur_p"] > 0:
        k, half = 9, 4
        apply = torch.rand(B, device=dev) < a["blur_p"]
        sig = torch.where(apply, _u(B, *a["blur_sigma"], dev), torch.full((B,), 1e-3, device=dev))
        xs = torch.arange(-half, half + 1, device=dev, dtype=torch.float32)
        ker = torch.exp(-(xs[None, :] ** 2) / (2 * sig[:, None] ** 2))
        ker = ker / ker.sum(1, keepdim=True)
        o = x.reshape(1, B, res, res)
        o = F.conv2d(F.pad(o, (half, half, 0, 0), mode="reflect"), ker.view(B, 1, 1, k), groups=B)
        o = F.conv2d(F.pad(o, (0, 0, half, half), mode="reflect"), ker.view(B, 1, k, 1), groups=B)
        x = o.reshape(B, 1, res, res)
    # gaussian noise
    if a["noise_p"] > 0:
        apply = (torch.rand(B, device=dev) < a["noise_p"]).float().view(B, 1, 1, 1)
        std = _u(B, 0.0, a["noise_std"], dev).view(B, 1, 1, 1) * apply
        x = (x + torch.randn_like(x) * std).clamp(0, 1)
    return x

def prep_eval(xb, res):
    x = xb.float().div_(255.0).unsqueeze(1)
    if x.shape[-1] != res:
        x = F.interpolate(x, size=(res, res), mode="bilinear", align_corners=False, antialias=True)
    return x

# ---- save example pairs of augmented views for manual inspection (plan requirement) ----
import matplotlib
import matplotlib.pyplot as plt
_arr = get_array("train")
_sel = np.sort(np.random.RandomState(0).choice(PRE_IDX, size=min(8, len(PRE_IDX)), replace=False))
_xb = torch.from_numpy(np.ascontiguousarray(_arr[_sel])).to(DEVICE)
with torch.no_grad():
    _v1, _v2 = gpu_augment(_xb, RES_IN, AUG_SSL), gpu_augment(_xb, RES_IN, AUG_SSL)
    _o = prep_eval(_xb.clone(), RES_IN)
assert torch.isfinite(_v1).all() and _v1.shape == (len(_sel), 1, RES_IN, RES_IN)
fig, ax = plt.subplots(3, len(_sel), figsize=(1.8 * len(_sel), 5.6))
ax = np.atleast_2d(ax).reshape(3, -1)
for j in range(len(_sel)):
    for r_, (img, t) in enumerate([(_o, "original"), (_v1, "view 1"), (_v2, "view 2")]):
        ax[r_, j].imshow(img[j, 0].cpu(), cmap="gray", vmin=0, vmax=1); ax[r_, j].axis("off")
        if j == 0: ax[r_, j].set_title(t, fontsize=8, loc="left")
plt.tight_layout(); plt.savefig(ART / "reports" / "augmentation_preview.png", dpi=110); plt.show()
print("Look at these: anatomy should stay recognisable (no flips, no heavy crops).")

In [ ]:
# ---- encoder + the five SSL methods ----
class SplitBatchNorm(nn.BatchNorm2d):
    # Single-GPU stand-in for MoCo's shuffle-BN: batch statistics are computed over `num_splits` sub-groups
    # so the query/key encoders cannot leak information through shared batch statistics.
    def __init__(self, num_features, num_splits, **kw):
        super().__init__(num_features, **kw)
        self.num_splits = num_splits
    def forward(self, x):
        N, C, H, W = x.shape
        if self.training and N % self.num_splits == 0:
            rm = self.running_mean.repeat(self.num_splits)
            rv = self.running_var.repeat(self.num_splits)
            out = F.batch_norm(x.reshape(-1, C * self.num_splits, H, W), rm, rv,
                               self.weight.repeat(self.num_splits), self.bias.repeat(self.num_splits),
                               True, self.momentum, self.eps).view(N, C, H, W)
            self.running_mean.data.copy_(rm.view(self.num_splits, C).mean(0))
            self.running_var.data.copy_(rv.view(self.num_splits, C).mean(0))
            return out
        return F.batch_norm(x, self.running_mean, self.running_var, self.weight, self.bias, False, self.momentum, self.eps)

class Backbone(nn.Module):
    MEAN, STD = 0.449, 0.226
    def __init__(self, arch="resnet18", norm_layer=None, imagenet=False):
        super().__init__()
        kw = {}
        if norm_layer is not None: kw["norm_layer"] = norm_layer
        kw["weights"] = torchvision.models.get_model_weights(arch).DEFAULT if imagenet else None
        net = getattr(torchvision.models, arch)(**kw)
        self.feat_dim = net.fc.in_features
        net.fc = nn.Identity()
        self.net = net
    def forward(self, x):                      # x: (B,1,H,W) in [0,1]; grayscale -> 3 identical channels
        x = (x - self.MEAN) / self.STD
        return self.net(x.expand(-1, 3, -1, -1))

def mlp(i, h, o):
    return nn.Sequential(nn.Linear(i, h, bias=False), nn.BatchNorm1d(h), nn.ReLU(inplace=True), nn.Linear(h, o))

def nt_xent(z1, z2, t):
    B = z1.shape[0]
    z = torch.cat([z1, z2])
    sim = (z @ z.t()) / t
    sim = sim.masked_fill(torch.eye(2 * B, dtype=torch.bool, device=z.device), -1e9)
    tgt = torch.cat([torch.arange(B, 2 * B), torch.arange(0, B)]).to(z.device)
    return F.cross_entropy(sim, tgt)

def info_nce(a, b, t):                           # a, b L2-normalised; positives on the diagonal
    return F.cross_entropy((a @ b.t()) / t, torch.arange(a.shape[0], device=a.device))

@torch.no_grad()
def ema_update(target, online, m):
    for pt, po in zip(target.parameters(), online.parameters()):
        pt.mul_(m).add_(po.detach(), alpha=1 - m)

def freeze(*mods):
    for mod in mods:
        for p in mod.parameters(): p.requires_grad = False

class SSLMethod(nn.Module):
    def after_step(self, progress): pass

class SimCLR(SSLMethod):
    def __init__(self, arch, temperature, dim=128, hidden=1024):
        super().__init__()
        self.encoder = Backbone(arch); self.proj = mlp(self.encoder.feat_dim, hidden, dim); self.t = temperature
    def forward(self, x1, x2, progress=0.0):
        B = x1.shape[0]
        z = F.normalize(self.proj(self.encoder(torch.cat([x1, x2]))).float(), dim=1)
        return nt_xent(z[:B], z[B:], self.t)

class MoCo(SSLMethod):                          # MoCo v2: MLP head, momentum encoder, queue, symmetric loss
    def __init__(self, arch, temperature, queue_size, momentum, bn_splits, dim=128, hidden=1024):
        super().__init__()
        self.encoder = Backbone(arch, norm_layer=lambda c: SplitBatchNorm(c, bn_splits))
        self.proj = mlp(self.encoder.feat_dim, hidden, dim)
        self.k_encoder, self.k_proj = copy.deepcopy(self.encoder), copy.deepcopy(self.proj)
        freeze(self.k_encoder, self.k_proj)
        self.t, self.m, self.K = temperature, momentum, queue_size
        self.register_buffer("queue", F.normalize(torch.randn(queue_size, dim), dim=1))
        self.register_buffer("ptr", torch.zeros(1, dtype=torch.long))
    @torch.no_grad()
    def _keys(self, x):
        perm = torch.randperm(x.shape[0], device=x.device)       # shuffle, encode, un-shuffle
        k = F.normalize(self.k_proj(self.k_encoder(x[perm])).float(), dim=1)
        return k[torch.argsort(perm)]
    @torch.no_grad()
    def _enqueue(self, k):
        n, p = k.shape[0], int(self.ptr)
        self.queue[(p + torch.arange(n, device=k.device)) % self.K] = k
        self.ptr[0] = (p + n) % self.K
    def _loss(self, q, k, queue):
        logits = torch.cat([(q * k).sum(1, keepdim=True), q @ queue.t()], 1) / self.t
        return F.cross_entropy(logits, torch.zeros(q.shape[0], dtype=torch.long, device=q.device))
    def forward(self, x1, x2, progress=0.0):
        queue = self.queue.clone()
        q1 = F.normalize(self.proj(self.encoder(x1)).float(), dim=1)
        q2 = F.normalize(self.proj(self.encoder(x2)).float(), dim=1)
        k1, k2 = self._keys(x1), self._keys(x2)
        loss = 0.5 * (self._loss(q1, k2, queue) + self._loss(q2, k1, queue))
        self._enqueue(torch.cat([k1, k2]))
        return loss
    def after_step(self, progress):
        ema_update(self.k_encoder, self.encoder, self.m); ema_update(self.k_proj, self.proj, self.m)

class BYOL(SSLMethod):
    def __init__(self, arch, tau_base, dim=128, hidden=1024):
        super().__init__()
        self.encoder = Backbone(arch)
        self.proj, self.pred = mlp(self.encoder.feat_dim, hidden, dim), mlp(dim, hidden, dim)
        self.t_encoder, self.t_proj = copy.deepcopy(self.encoder), copy.deepcopy(self.proj)
        freeze(self.t_encoder, self.t_proj)
        self.tau_base = tau_base
    def _online(self, x): return self.pred(self.proj(self.encoder(x)))
    @torch.no_grad()
    def _target(self, x): return self.t_proj(self.t_encoder(x))
    @staticmethod
    def _l(p, t):
        return (2 - 2 * (F.normalize(p.float(), dim=1) * F.normalize(t.float(), dim=1)).sum(1)).mean()
    def forward(self, x1, x2, progress=0.0):
        p1, p2 = self._online(x1), self._online(x2)
        t1, t2 = self._target(x1), self._target(x2)
        return self._l(p1, t2) + self._l(p2, t1)
    def after_step(self, progress):
        tau = 1 - (1 - self.tau_base) * (math.cos(math.pi * progress) + 1) / 2
        ema_update(self.t_encoder, self.encoder, tau); ema_update(self.t_proj, self.proj, tau)

class NNCLR(SSLMethod):                          # nearest neighbour from a support set is the positive
    def __init__(self, arch, temperature, queue_size, dim=128, hidden=1024):
        super().__init__()
        self.encoder = Backbone(arch)
        f = self.encoder.feat_dim
        self.proj = nn.Sequential(nn.Linear(f, hidden, bias=False), nn.BatchNorm1d(hidden), nn.ReLU(inplace=True),
                                  nn.Linear(hidden, hidden, bias=False), nn.BatchNorm1d(hidden), nn.ReLU(inplace=True),
                                  nn.Linear(hidden, dim))
        self.pred = mlp(dim, hidden, dim)
        self.t, self.K = temperature, queue_size
        self.register_buffer("support", F.normalize(torch.randn(queue_size, dim), dim=1))
        self.register_buffer("filled", torch.zeros(1, dtype=torch.long))
        self.register_buffer("ptr", torch.zeros(1, dtype=torch.long))
    @torch.no_grad()
    def _nn(self, z):
        if int(self.filled) < self.K: return z                  # support set not full yet: plain two-view positive
        return self.support[(z @ self.support.t()).argmax(1)]
    @torch.no_grad()
    def _enqueue(self, z):
        n, p = z.shape[0], int(self.ptr)
        self.support[(p + torch.arange(n, device=z.device)) % self.K] = z
        self.ptr[0] = (p + n) % self.K
        self.filled[0] = min(self.K, int(self.filled) + n)
    def forward(self, x1, x2, progress=0.0):
        h1, h2 = self.proj(self.encoder(x1)).float(), self.proj(self.encoder(x2)).float()
        z1, z2 = F.normalize(h1.detach(), dim=1), F.normalize(h2.detach(), dim=1)
        p1, p2 = F.normalize(self.pred(h1).float(), dim=1), F.normalize(self.pred(h2).float(), dim=1)
        loss = 0.5 * (info_nce(self._nn(z1), p2, self.t) + info_nce(self._nn(z2), p1, self.t))
        self._enqueue(torch.cat([z1, z2]))
        return loss

@torch.no_grad()
def sinkhorn(scores, eps=0.05, iters=3):
    Q = torch.exp(scores / eps).t()                             # (K, B)
    Q /= Q.sum()
    K, B = Q.shape
    for _ in range(iters):
        Q /= Q.sum(dim=1, keepdim=True); Q /= K
        Q /= Q.sum(dim=0, keepdim=True); Q /= B
    return (Q * B).t()

class SwAV(SSLMethod):                          # two global views only (no multi-crop), no queue
    def __init__(self, arch, temperature, n_prototypes, dim=128, hidden=1024, eps=0.05, iters=3):
        super().__init__()
        self.encoder = Backbone(arch); self.proj = mlp(self.encoder.feat_dim, hidden, dim)
        self.prototypes = nn.Parameter(F.normalize(torch.randn(n_prototypes, dim), dim=1))
        self.t, self.eps, self.iters, self.freeze_steps = temperature, eps, iters, 0
        self.register_buffer("step", torch.zeros(1, dtype=torch.long))
    def forward(self, x1, x2, progress=0.0):
        with torch.no_grad(): self.prototypes.copy_(F.normalize(self.prototypes, dim=1))
        w = self.prototypes.detach() if int(self.step) < self.freeze_steps else self.prototypes
        B = x1.shape[0]
        z = F.normalize(self.proj(self.encoder(torch.cat([x1, x2]))).float(), dim=1)
        s = z @ w.float().t()
        s1, s2 = s[:B], s[B:]
        q1, q2 = sinkhorn(s1.detach(), self.eps, self.iters), sinkhorn(s2.detach(), self.eps, self.iters)
        l1 = -(q1 * F.log_softmax(s2 / self.t, dim=1)).sum(1).mean()
        l2 = -(q2 * F.log_softmax(s1 / self.t, dim=1)).sum(1).mean()
        return 0.5 * (l1 + l2)
    def after_step(self, progress):
        self.step += 1

SSL_METHODS = ["simclr", "moco", "byol", "nnclr", "swav"]

def build_ssl(name):
    c, arch = CFG["ssl"], CFG["arch"]
    if name == "simclr": return SimCLR(arch, c["temperature"]["simclr"])
    if name == "moco":   return MoCo(arch, c["temperature"]["moco"], c["moco_queue"], c["moco_momentum"], c["moco_bn_splits"])
    if name == "byol":   return BYOL(arch, c["byol_tau"])
    if name == "nnclr":  return NNCLR(arch, c["temperature"]["nnclr"], c["nnclr_queue"])
    if name == "swav":   return SwAV(arch, c["temperature"]["swav"], c["swav_prototypes"])
    raise ValueError(name)
print("methods:", SSL_METHODS)

In [ ]:
# ---- SSL pretraining loop (AMP, cosine LR with warm-up, per-epoch checkpoint, auto-resume) ----
def prefetch(gen, depth=3):
    q, end = queue.Queue(maxsize=depth), object()
    def worker():
        try:
            for item in gen: q.put(item)
            q.put(end)
        except Exception as e:
            q.put(e)
    threading.Thread(target=worker, daemon=True).start()
    while True:
        item = q.get()
        if item is end: return
        if isinstance(item, Exception): raise item
        yield item

def ssl_batches(arr, idx, bs, epoch, seed, max_steps=None):
    perm = np.random.RandomState(seed * 1000 + epoch).permutation(idx)
    n = len(perm) // bs
    if max_steps: n = min(n, max_steps)
    for i in range(n):
        yield torch.from_numpy(np.ascontiguousarray(arr[np.sort(perm[i * bs:(i + 1) * bs])]))

def lr_at(step, total, warm, base):
    if step < warm: return base * (step + 1) / max(1, warm)
    p = (step - warm) / max(1, total - warm)
    return base * (0.001 + 0.999 * 0.5 * (1 + math.cos(math.pi * p)))

def make_scaler():
    try: return torch.amp.GradScaler("cuda", enabled=USE_AMP)
    except Exception: return torch.cuda.amp.GradScaler(enabled=USE_AMP)

@torch.no_grad()
def feature_spread(encoder, arr, idx, n=512):
    # collapse monitor: ~0 means all images map to the same feature vector; healthy training stays well above 0
    encoder.eval()
    sel = np.sort(np.random.RandomState(0).choice(idx, size=min(n, len(idx)), replace=False))
    x = prep_eval(torch.from_numpy(np.ascontiguousarray(arr[sel])).to(DEVICE), RES_IN)
    f = F.normalize(encoder(x).float(), dim=1)
    encoder.train()
    return float(f.std(dim=0).mean() * math.sqrt(f.shape[1]))

def run_pretrain(name, epochs=None, max_steps=None, tag=None, batch_size=None):
    c = CFG["ssl"]
    epochs, bs = epochs or c["epochs"], batch_size or c["batch_size"]
    out = ART / "pretrain" / (tag or name); out.mkdir(parents=True, exist_ok=True)
    if (out / "done.json").exists():
        print(f"[{name}] already finished -> {out}"); return json.loads((out / "done.json").read_text())
    if name == "moco": assert bs % c["moco_bn_splits"] == 0
    set_seed(c["seed"])
    arr = get_array("train")
    method = build_ssl(name).to(DEVICE)
    spe = len(PRE_IDX) // bs
    if max_steps: spe = min(spe, max_steps)
    assert spe >= 1, "pretraining pool smaller than one batch"
    if hasattr(method, "freeze_steps"): method.freeze_steps = spe        # SwAV: prototypes frozen for the first epoch
    total, base = epochs * spe, c["lr"] * bs / 256
    params = [p for p in method.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=base, weight_decay=c["weight_decay"])
    scaler = make_scaler()
    (out / "config.json").write_text(json.dumps(dict(method=name, epochs=epochs, batch_size=bs, steps_per_epoch=spe,
        arch=CFG["arch"], train_res=RES_IN, aug=AUG_SSL, ssl=c, pretrain_images=int(len(PRE_IDX)),
        torch=torch.__version__), indent=1, default=str))
    start, log = 0, []
    if (out / "last.pt").exists():
        ck = torch.load(out / "last.pt", map_location=DEVICE, weights_only=False)
        method.load_state_dict(ck["method"]); opt.load_state_dict(ck["opt"]); scaler.load_state_dict(ck["scaler"])
        start, log = ck["epoch"], ck["log"]
        print(f"[{name}] resumed at epoch {start}/{epochs}")
    method.train()
    sec_per_step, bad = None, 0
    for ep in range(start, epochs):
        t0, tot, cnt = time.time(), torch.zeros((), device=DEVICE), 0
        for i, xb in enumerate(prefetch(ssl_batches(arr, PRE_IDX, bs, ep, c["seed"], max_steps))):
            step = ep * spe + i
            for g in opt.param_groups: g["lr"] = lr_at(step, total, c["warmup_epochs"] * spe, base)
            xb = xb.to(DEVICE, non_blocking=True)
            with torch.no_grad():
                v1, v2 = gpu_augment(xb, RES_IN, AUG_SSL), gpu_augment(xb, RES_IN, AUG_SSL)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
                loss = method(v1, v2, progress=step / total)
            if not torch.isfinite(loss):
                bad += 1; opt.zero_grad(set_to_none=True)
                assert bad <= 20, f"[{name}] loss is non-finite repeatedly -> stopping"
                continue
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            method.after_step(step / total)
            tot += loss.detach(); cnt += 1
        if DEVICE == "cuda": torch.cuda.synchronize()
        dt = time.time() - t0
        sec_per_step = dt / max(1, spe)
        row = dict(epoch=ep + 1, loss=float(tot / max(1, cnt)), feature_spread=feature_spread(method.encoder, arr, PRE_IDX),
                   seconds=round(dt, 1), lr=opt.param_groups[0]["lr"])
        log.append(row); print(f"[{name}] {row}")
        tmp = out / "last.tmp"
        torch.save(dict(method=method.state_dict(), opt=opt.state_dict(), scaler=scaler.state_dict(), epoch=ep + 1, log=log), tmp)
        tmp.replace(out / "last.pt")
    torch.save(method.encoder.state_dict(), out / "encoder.pt")
    pd.DataFrame(log).to_csv(out / "log.csv", index=False)
    done = dict(method=name, epochs=epochs, final_loss=log[-1]["loss"] if log else None,
                sec_per_step=sec_per_step, steps_per_epoch=spe, bad_steps=bad)
    (out / "done.json").write_text(json.dumps(done))
    if (out / "last.pt").exists(): (out / "last.pt").unlink()          # free Drive space
    return done

In [ ]:
# ---- SMOKE TEST: every SSL method, a few steps each. Run this before anything marked [FULL]. ----
SMOKE_BS = 64
smoke = {}
for name in SSL_METHODS:
    shutil.rmtree(ART / "pretrain" / f"smoke_{name}", ignore_errors=True)
    smoke[name] = run_pretrain(name, epochs=2, max_steps=4, tag=f"smoke_{name}", batch_size=SMOKE_BS)
    assert smoke[name]["final_loss"] is not None and math.isfinite(smoke[name]["final_loss"]), name
full_spe = len(PRE_IDX) // CFG["ssl"]["batch_size"]
print("\nmethod  final_loss  sec/step(bs=%d)  est. minutes per FULL epoch at bs=%d" % (SMOKE_BS, CFG["ssl"]["batch_size"]))
for n, d in smoke.items():
    est = d["sec_per_step"] * (CFG["ssl"]["batch_size"] / SMOKE_BS) * full_spe / 60
    print(f"{n:7s} {d['final_loss']:.4f}     {d['sec_per_step']:.3f}            ~{est:.1f}   (rough: first steps include warm-up overhead)")

### Sizing the pretraining run
Use the estimate above: `epochs × minutes/epoch × 5 methods` is your total GPU time, and Colab can disconnect at any point, so each method checkpoints every epoch and resumes by itself. Use **the same epoch count for all five methods** (fairness). If time is short, reduce `CFG["ssl"]["epochs"]` or `train_res` in the config cell, not per method.

In [ ]:
# [FULL] pretrain all five methods (resumable: re-run this cell after a disconnect)
for name in SSL_METHODS:
    run_pretrain(name)

In [ ]:
# ---- downstream: metrics ----
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve

def _clean(o):
    if isinstance(o, dict): return {str(k): _clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_clean(v) for v in o]
    if isinstance(o, (np.floating, float)): return None if not math.isfinite(float(o)) else float(o)
    if isinstance(o, np.integer): return int(o)
    return o

def macro_auroc(y, p):
    v = [roc_auc_score(y[:, j], p[:, j]) for j in range(y.shape[1]) if y[:, j].min() != y[:, j].max()]
    return float(np.mean(v)) if v else float("nan")

def youden_thresholds(y, p):
    th = []
    for j in range(y.shape[1]):
        if y[:, j].min() == y[:, j].max(): th.append(0.5); continue
        fpr, tpr, t = roc_curve(y[:, j], p[:, j])
        k = int(np.argmax(tpr - fpr))
        th.append(float(t[k]) if np.isfinite(t[k]) else 0.5)
    return th

def compute_metrics(y, p, thresholds=None):
    res = {"auroc": {}, "auprc": {}}
    for j, l in enumerate(LABELS):
        one_class = y[:, j].min() == y[:, j].max()
        res["auroc"][l] = float("nan") if one_class else float(roc_auc_score(y[:, j], p[:, j]))
        res["auprc"][l] = float("nan") if one_class else float(average_precision_score(y[:, j], p[:, j]))
    res["macro_auroc"] = float(np.nanmean(list(res["auroc"].values()))) if not all(math.isnan(v) for v in res["auroc"].values()) else float("nan")
    res["macro_auprc"] = float(np.nanmean(list(res["auprc"].values()))) if not all(math.isnan(v) for v in res["auprc"].values()) else float("nan")
    if thresholds is not None:
        res["threshold_metrics"] = {}
        for j, l in enumerate(LABELS):
            pred, yt = p[:, j] >= thresholds[j], y[:, j] == 1
            tp, fn = int((pred & yt).sum()), int((~pred & yt).sum())
            tn, fp = int((~pred & ~yt).sum()), int((pred & ~yt).sum())
            sens = tp / (tp + fn) if tp + fn else float("nan"); spec = tn / (tn + fp) if tn + fp else float("nan")
            f1 = 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) else float("nan")
            res["threshold_metrics"][l] = dict(threshold=thresholds[j], sensitivity=sens, specificity=spec,
                                               f1=f1, balanced_accuracy=np.nanmean([sens, spec]) if not (math.isnan(sens) and math.isnan(spec)) else float("nan"))
    return res

def bootstrap_ci(y, p, patient_ids, n_boot=1000, seed=0):
    rng = np.random.RandomState(seed)
    groups = {}
    for i, pid in enumerate(patient_ids): groups.setdefault(pid, []).append(i)
    pids = list(groups); macros, per = [], {l: [] for l in LABELS}
    for _ in range(n_boot):
        rows = np.concatenate([groups[pids[k]] for k in rng.randint(0, len(pids), len(pids))])
        yb, pb = y[rows], p[rows]
        macros.append(macro_auroc(yb, pb))
        for j, l in enumerate(LABELS):
            if yb[:, j].min() != yb[:, j].max(): per[l].append(roc_auc_score(yb[:, j], pb[:, j]))
    q = lambda v: [float(np.nanpercentile(v, 2.5)), float(np.nanpercentile(v, 97.5))] if len(v) else [float("nan")] * 2
    return dict(macro_auroc=q(macros), per_label={l: q(v) for l, v in per.items()})

# tests on tiny synthetic examples
_y = np.array([[0], [0], [1], [1]], dtype=np.float32); _p = np.array([[0.1], [0.4], [0.35], [0.8]])
assert abs(roc_auc_score(_y[:, 0], _p[:, 0]) - 0.75) < 1e-9
assert math.isnan(macro_auroc(np.zeros((4, 1), dtype=np.float32), _p))        # single-class label handled
print("metric tests passed")

In [ ]:
# ---- downstream: models and training ----
class Classifier(nn.Module):
    def __init__(self, backbone, n_out):
        super().__init__()
        self.backbone, self.head = backbone, nn.Linear(backbone.feat_dim, n_out)
    def forward(self, x): return self.head(self.backbone(x))

def build_backbone(init, pretrain_tag=None):
    # init: "scratch" | "imagenet" | one of SSL_METHODS
    if init == "imagenet": return Backbone(CFG["arch"], imagenet=True)       # downloads torchvision weights
    bb = Backbone(CFG["arch"])
    if init == "scratch": return bb
    p = ART / "pretrain" / (pretrain_tag or init) / "encoder.pt"
    assert p.exists(), f"no pretrained encoder at {p} - run pretraining for '{init}' first"
    bb.load_state_dict(torch.load(p, map_location="cpu"))
    return bb

def sd_to_half(sd):    # smaller checkpoints on Drive; BN running stats stay float32
    return {k: (v.half() if v.is_floating_point() and not k.endswith(("running_mean", "running_var")) else v) for k, v in sd.items()}
def sd_to_float(sd):
    return {k: (v.float() if v.is_floating_point() else v) for k, v in sd.items()}

def train_batches(arr, a_idx, Y, bs, rng):
    perm = rng.permutation(len(a_idx))
    for i in range(0, len(perm), bs):
        pos = np.sort(perm[i:i + bs])
        yield torch.from_numpy(np.ascontiguousarray(arr[a_idx[pos]])), torch.from_numpy(Y[pos])

@torch.no_grad()
def predict_logits(model, arr, a_idx, bs=256):
    model.eval(); out = []
    for i in range(0, len(a_idx), bs):
        x = prep_eval(torch.from_numpy(np.ascontiguousarray(arr[a_idx[i:i + bs]])).to(DEVICE), RES_IN)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
            out.append(model(x).float().cpu())
    return torch.cat(out).numpy()

@torch.no_grad()
def extract_features(backbone, arr, a_idx, bs=256):
    backbone.eval(); out = []
    for i in range(0, len(a_idx), bs):
        x = prep_eval(torch.from_numpy(np.ascontiguousarray(arr[a_idx[i:i + bs]])).to(DEVICE), RES_IN)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
            out.append(backbone(x).float())
    return torch.cat(out)

sigmoid = lambda z: 1 / (1 + np.exp(-z))

def fit_finetune(model, arr, tr_idx, Ytr, dv_idx, Ydv, pw, epochs, lr_bb, lr_head, seed):
    d = CFG["down"]
    opt = torch.optim.AdamW([{"params": model.backbone.parameters(), "lr": lr_bb},
                             {"params": model.head.parameters(), "lr": lr_head}], weight_decay=d["weight_decay"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, epochs))
    scaler, lossf = make_scaler(), nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pw, device=DEVICE))
    rng, best, best_ep, best_state, bad, hist = np.random.RandomState(seed), -1.0, -1, None, 0, []
    for ep in range(epochs):
        model.train(); tot, n = 0.0, 0
        for xb, yb in prefetch(train_batches(arr, tr_idx, Ytr, d["batch_size"], rng)):
            x = gpu_augment(xb.to(DEVICE), RES_IN, AUG_DOWN); y = yb.to(DEVICE)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
                loss = lossf(model(x).float(), y)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            tot += loss.item() * len(yb); n += len(yb)
        sched.step()
        auc = macro_auroc(Ydv, sigmoid(predict_logits(model, arr, dv_idx)))
        hist.append(dict(epoch=ep + 1, train_loss=tot / max(1, n), dev_macro_auroc=auc))
        score = auc if math.isfinite(auc) else -1.0
        if score > best:
            best, best_ep, bad = score, ep + 1, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= d["patience"]: break
    model.load_state_dict(best_state)
    return {"model": sd_to_half(best_state)}, hist, best_ep

def fit_linear(model, arr, tr_idx, Ytr, dv_idx, Ydv, pw, epochs, seed, bs=512, lr=1e-3):
    d = CFG["down"]
    ftr, fdv = extract_features(model.backbone, arr, tr_idx), extract_features(model.backbone, arr, dv_idx)
    mu, sd = ftr.mean(0), ftr.std(0).clamp_min(1e-6)
    ftr, fdv = (ftr - mu) / sd, (fdv - mu) / sd
    g = torch.Generator(device=DEVICE); g.manual_seed(seed)
    head = nn.Linear(ftr.shape[1], len(LABELS)).to(DEVICE)
    opt = torch.optim.AdamW(head.parameters(), lr=lr, weight_decay=d["weight_decay"])
    lossf, Yt = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pw, device=DEVICE)), torch.from_numpy(Ytr).to(DEVICE)
    best, best_ep, best_head, bad, hist = -1.0, -1, None, 0, []
    for ep in range(epochs):
        perm = torch.randperm(len(ftr), device=DEVICE, generator=g)
        for i in range(0, len(perm), bs):
            b = perm[i:i + bs]
            loss = lossf(head(ftr[b]), Yt[b]); opt.zero_grad(); loss.backward(); opt.step()
        with torch.no_grad(): auc = macro_auroc(Ydv, torch.sigmoid(head(fdv)).cpu().numpy())
        hist.append(dict(epoch=ep + 1, dev_macro_auroc=auc))
        score = auc if math.isfinite(auc) else -1.0
        if score > best:
            best, best_ep, bad = score, ep + 1, 0
            best_head = {k: v.detach().clone() for k, v in head.state_dict().items()}
        else:
            bad += 1
            if bad >= d["linear_patience"]: break
    W, b0 = best_head["weight"], best_head["bias"]                       # fold the standardisation into the head
    model.head.weight.data.copy_(W / sd); model.head.bias.data.copy_(b0 - (W * (mu / sd)).sum(1))
    return {"head": {k: v.cpu() for k, v in model.head.state_dict().items()}}, hist, best_ep

def run_downstream(init, protocol, budget, seed, root="downstream", pretrain_tag=None, max_epochs=None):
    assert protocol in ("finetune", "linear")
    name = f"{init}_{protocol}"; run_id = f"{name}_b{budget:g}_s{seed}"
    out = ART / root / run_id; out.mkdir(parents=True, exist_ok=True)
    if (out / "metrics.json").exists():
        return json.loads((out / "metrics.json").read_text())
    d, t0 = CFG["down"], time.time()
    set_seed(seed)
    if DEVICE == "cuda": torch.cuda.reset_peak_memory_stats()
    order = SPLITS["orders"][str(seed)]
    pats = set(order[:n_budget(budget, len(order))])
    tr = TRAIN_F[TRAIN_F.patient_id.isin(pats)]
    tr_idx, Ytr = tr.arr_idx.values, tr[Y_COLS].values.astype(np.float32)
    dv_idx, Ydv = DEV_F.arr_idx.values, DEV_F[Y_COLS].values.astype(np.float32)
    pos = Ytr.sum(0)
    pw = np.clip((len(Ytr) - pos) / np.maximum(pos, 1), 1.0, d["pos_weight_max"]).astype(np.float32)
    arr = get_array("train")
    model = Classifier(build_backbone(init, pretrain_tag), len(LABELS)).to(DEVICE)
    if protocol == "linear":
        epochs = max_epochs or d["linear_epochs"]
        ck, hist, best_ep = fit_linear(model, arr, tr_idx, Ytr, dv_idx, Ydv, pw, epochs, seed)
        lrs = dict(head=1e-3)
    else:
        epochs = max_epochs or d["max_epochs"][budget]
        lr_bb = d["lr_backbone_scratch"] if init == "scratch" else d["lr_backbone_pretrained"]
        ck, hist, best_ep = fit_finetune(model, arr, tr_idx, Ytr, dv_idx, Ydv, pw, epochs, lr_bb, d["lr_head"], seed)
        lrs = dict(backbone=lr_bb, head=d["lr_head"])
    torch.save(ck, out / "best.pt")
    p_dv = sigmoid(predict_logits(model, arr, dv_idx))
    th = youden_thresholds(Ydv, p_dv)                                   # thresholds chosen on the dev split only
    dev = compute_metrics(Ydv, p_dv, th)
    cfg = dict(run_id=run_id, name=name, init=init, protocol=protocol, budget=budget, seed=seed, pretrain_tag=pretrain_tag,
               n_patients=len(pats), n_images=int(len(tr_idx)), pos_weight=pw.tolist(), labels=LABELS,
               uncertain_to_one=CFG["uncertain_to_one"], arch=CFG["arch"], train_res=RES_IN, aug=AUG_DOWN, lrs=lrs,
               max_epochs=epochs, torch=torch.__version__, device=DEVICE)
    (out / "config.json").write_text(json.dumps(_clean(cfg), indent=1))
    metrics = dict(run_id=run_id, name=name, budget=budget, seed=seed, dev=dev, thresholds=th, best_epoch=best_ep,
                   epochs_run=len(hist), seconds=round(time.time() - t0, 1),
                   peak_gpu_gb=(torch.cuda.max_memory_allocated() / 2**30 if DEVICE == "cuda" else None),
                   ckpt_mb=round((out / "best.pt").stat().st_size / 2**20, 1), history=hist)
    (out / "metrics.json").write_text(json.dumps(_clean(metrics), indent=1))
    print(f"{run_id}: dev macro AUROC {dev['macro_auroc']:.4f} | epochs {len(hist)} (best {best_ep}) | {metrics['seconds']}s | n_img {len(tr_idx)}")
    return json.loads((out / "metrics.json").read_text())

In [ ]:
# ---- SMOKE TEST: downstream pipeline end to end on the smoke SSL checkpoints (separate output folder) ----
shutil.rmtree(ART / "downstream_smoke", ignore_errors=True)
S = dict(budget=0.05, seed=CFG["seeds"][0], root="downstream_smoke", max_epochs=2)
run_downstream("scratch", "finetune", **S)
run_downstream("simclr", "linear", pretrain_tag="smoke_simclr", **S)
run_downstream("moco", "finetune", pretrain_tag="smoke_moco", **S)
try:
    run_downstream("imagenet", "finetune", **S)
except Exception as e:                      # needs internet for torchvision weights
    print("ImageNet baseline smoke test failed:", repr(e)[:300])
# nesting + determinism check for the label budgets
o = SPLITS["orders"][str(CFG["seeds"][0])]
assert set(o[:n_budget(0.01, len(o))]) <= set(o[:n_budget(0.05, len(o))]) <= set(o[:n_budget(1.0, len(o))])
print("smoke test finished")

### Main experiment grid
Methods: `scratch`, `imagenet` (reference baselines) and the five SSL encoders, each as `finetune`; SSL encoders also as `linear` probe. Order below is smallest budget first. Finished runs are skipped, so you can interrupt and resume. Check the per-run time on the first few runs before launching the largest budget. If compute is tight, use the documented fallbacks: fewer seeds, or drop the 25% budget.

In [ ]:
# [FULL] label-budget sweep. Edit the lists to run a subset (e.g. seeds=[0] first).
SWEEP_SEEDS   = CFG["seeds"]
SWEEP_BUDGETS = CFG["budgets"]
SWEEP_RUNS = ([("scratch", "finetune"), ("imagenet", "finetune")]
              + [(m, "linear") for m in SSL_METHODS] + [(m, "finetune") for m in SSL_METHODS])
for budget in SWEEP_BUDGETS:
    for seed in SWEEP_SEEDS:
        for init, protocol in SWEEP_RUNS:
            run_downstream(init, protocol, budget, seed)

In [ ]:
# ---- aggregate straight from the raw metric files (dev split) ----
def collect(root="downstream"):
    rows = []
    for mp in sorted((ART / root).glob("*/metrics.json")):
        m = json.loads(mp.read_text())
        rows.append(dict(method=m["name"], budget=m["budget"], seed=m["seed"], macro_auroc=m["dev"]["macro_auroc"],
                         macro_auprc=m["dev"]["macro_auprc"], seconds=m["seconds"],
                         **{f"auroc_{l}": m["dev"]["auroc"][l] for l in LABELS}))
    return pd.DataFrame(rows)

RES_DF = collect()
if len(RES_DF):
    g = RES_DF.groupby(["method", "budget"]).macro_auroc.agg(["mean", "std", "count"]).reset_index()
    g["cell"] = g.apply(lambda r: f"{r['mean']:.3f} ± {0 if pd.isna(r['std']) else r['std']:.3f} (n={int(r['count'])})", axis=1)
    table = g.pivot(index="method", columns="budget", values="cell")
    display(table) if "display" in globals() else print(table)
    table.to_csv(ART / "reports" / "dev_macro_auroc_table.csv")
    fig, ax = plt.subplots(figsize=(7, 4.5))
    for meth, sub in g.groupby("method"):
        ax.errorbar(sub["budget"] * 100, sub["mean"], yerr=sub["std"].fillna(0), marker="o", capsize=3, label=meth)
    ax.set_xscale("log"); ax.set_xlabel("labelled patients (%)"); ax.set_ylabel("dev macro AUROC")
    ax.set_title("Label efficiency (internal dev split)"); ax.legend(fontsize=7, ncol=2); ax.grid(alpha=.3)
    plt.tight_layout(); plt.savefig(ART / "reports" / "dev_label_efficiency.png", dpi=130); plt.show()
else:
    print("no finished runs in", ART / "downstream")

### Final evaluation on the official validation set
Run this **once**, after the configurations are locked using the dev results above. The flag below must be set by hand. It reports AUROC/AUPRC with thresholds picked on the dev split and patient-level bootstrap 95% CIs. The official validation set is small (a few hundred images), so expect wide intervals.

In [ ]:
CONFIRM_FINAL = False        # set True only when you are ready to touch the official validation set
FINAL_RUNS = []              # e.g. ["simclr_finetune_b0.01_s0", "scratch_finetune_b0.01_s0"]

def load_trained(run_dir):
    cfg = json.loads((run_dir / "config.json").read_text())
    ck = torch.load(run_dir / "best.pt", map_location="cpu")
    if cfg["protocol"] == "finetune":
        model = Classifier(Backbone(CFG["arch"]), len(LABELS)); model.load_state_dict(sd_to_float(ck["model"]))
    else:
        model = Classifier(build_backbone(cfg["init"], cfg.get("pretrain_tag")), len(LABELS)); model.head.load_state_dict(ck["head"])
    return model.to(DEVICE)

def final_eval(run_ids, root="downstream", n_boot=1000):
    assert CONFIRM_FINAL, "Set CONFIRM_FINAL = True to evaluate on the official validation set."
    arr, rows = get_array("valid"), []
    y = VALID_F[Y_COLS].values.astype(np.float32)
    for rid in run_ids:
        rd = ART / root / rid
        m = json.loads((rd / "metrics.json").read_text())
        p = sigmoid(predict_logits(load_trained(rd), arr, VALID_F.arr_idx.values))
        res = compute_metrics(y, p, m["thresholds"]); res["ci95"] = bootstrap_ci(y, p, VALID_F.patient_id.values, n_boot)
        res["n_images"] = int(len(y)); res["n_patients"] = int(VALID_F.patient_id.nunique())
        (rd / "valid_metrics.json").write_text(json.dumps(_clean(res), indent=1))
        rows.append(dict(run=rid, macro_auroc=res["macro_auroc"], ci_low=res["ci95"]["macro_auroc"][0], ci_high=res["ci95"]["macro_auroc"][1]))
    return pd.DataFrame(rows)

if CONFIRM_FINAL and FINAL_RUNS:
    print(final_eval(FINAL_RUNS))
else:
    print("final evaluation skipped (CONFIRM_FINAL is False or FINAL_RUNS is empty)")

### Next (not in this notebook yet)
Once the dev results pick a winner among the five SSL methods: teacher–student stage (fine-tune the winner on a small budget → pseudo-label the unlabeled pool → train the student on real + confident pseudo-labels). Stretch tier: CheXzero / VLM track.

Provenance note for the report: data came from the Kaggle mirror `mimsadiislam/chexpert` (license listed as unknown there); the original release is Stanford's CheXpert under its Research Use Agreement (https://stanfordmlgroup.github.io/competitions/chexpert/).